In [ ]:
import pandas as pd
pd.set_option('display.max_columns', 1000)
Data = pd.read_csv('/Users/ganeshjai/Documents/GitHub/Pipeline/data/master_data.csv')
Data

In [ ]:
from preprocess import DataPreprocessor
import numpy as np
import pandas as pd

# Initialize
pre = DataPreprocessor(Data)

print("\n=== START PREPROCESSING ===")

# 1. Drop initial columns
drop_1 = [
    'pain_when', 'is_smoker', 'per_day', 'max', 'amount',
    'depression', 'anxiety', 'schizophrenia',
    'cocaine_inject_days', 'speedball_inject_days',
    'opioid_inject_days', 'speed_inject_days',
    'UDS_Alcohol_Count', 'UDS_Mdma/Hallucinogen_Count'
]
pre.drop_columns_and_return([c for c in drop_1 if c in pre.dataframe.columns])
print("Dropped initial columns")
display(pre.dataframe)


In [ ]:
# 2. Convert Yes/No to binary
pre.convert_yes_no_to_binary()
print("Converted Yes/No to binary")
display(pre.dataframe)


In [ ]:
# 3. Process TLFB columns
tlfb_keep = [
    'TLFB_Alcohol_Count', 'TLFB_Amphetamine_Count', 'TLFB_Cocaine_Count',
    'TLFB_Heroin_Count', 'TLFB_Benzodiazepine_Count', 'TLFB_Opioid_Count',
    'TLFB_THC_Count', 'TLFB_Methadone_Count', 'TLFB_Buprenorphine_Count'
]
pre.process_tlfb_columns(tlfb_keep)
print("Processed TLFB columns")
display(pre.dataframe.filter(regex="TLFB").head())


In [ ]:
# 4. Behavioral columns
pre.calculate_behavioral_columns()
print("Calculated behavioral columns")
display(pre.dataframe[['Sex','msm_npt','txx_prt','Homosexual_Behavior','Non_monogamous_Relationships']].head())


In [ ]:
# 6. Drop msm columns
drop_2 = ['msm_npt','msm_frq','txx_prt']
pre.drop_columns_and_return([c for c in drop_2 if c in pre.dataframe.columns])
print("Dropped msm columns")
display(pre.dataframe.head())


In [ ]:
# 7. Transform with NaN handling
pre.transform_data_with_nan_handling()
print("Applied transform_data_with_nan_handling")
display(pre.dataframe.head())


In [ ]:
# 8. Rename columns
pre.rename_columns()
print("Renamed columns")
display(pre.dataframe.head())


In [ ]:
# 9. Transform NaN → 0 for binary
pre.transform_nan_to_zero_for_binary_columns()
print("Transformed NaN to 0 for binary columns")
display(pre.dataframe.head())


In [ ]:
# 10. Transform and rename heroin_inject_days
pre.transform_and_rename_column('heroin_inject_days', 'rbsivheroin')
print("Transformed & renamed heroin_inject_days → rbsivheroin")
display(pre.dataframe.filter(regex="heroin|rbsivheroin").head())


In [ ]:
# 11. Fill NaN with 0 for ftnd
pre.fill_nan_with_zero('ftnd')
print("Filled NaN with 0 for ftnd")
display(pre.dataframe.filter(like='ftnd').head())


In [ ]:
# 12. Drop final columns
drop_3 = [
    'rbs_iv_days', 'race', 'RBS_cocaine_Days', 'RBS_heroin_Days',
    'RBS_opioid_Days', 'RBS_speed_Days', 'RBS_speedball_Days'
]
pre.drop_columns_and_return([c for c in drop_3 if c in pre.dataframe.columns])
print("Dropped final columns")
display(pre.dataframe)


In [ ]:
# 13. Convert UDS counts to binary
pre.convert_uds_to_binary()
print("Converted UDS counts to binary")
display(pre.dataframe.filter(regex="UDS").head())


In [ ]:
print("=== PREPROCESSING COMPLETE ===")
display(pre.dataframe)


In [ ]:
# Save to CSV
pre.dataframe.to_csv("/Users/ganeshjai/Documents/GitHub/Pipeline/data/master_data_clean.csv", index=False)


In [ ]:
pd.read_csv('/Users/ganeshjai/Documents/GitHub/Pipeline/data/master_data_clean.csv')
df = pd.read_csv('/Users/ganeshjai/Documents/GitHub/Pipeline/data/master_data_clean.csv')
df

In [ ]:
# exclude missing race (0)
df = df[df["RaceEth"] != 0].copy()

# create is_minority
df["is_minority"] = np.where(df["RaceEth"] == 1, 0, 1)

# ensure covariates exist/are clean
df["age"] = pd.to_numeric(df["age"], errors="coerce")
if "is_female" not in df.columns:
    raise ValueError("Missing 'is_female' — add during preprocessing (0=male,1=female).")

# keep only rows usable for matching
needed = ["who","age","is_female","is_minority"]
df = df.dropna(subset=needed).reset_index(drop=True)

print("Counts → majority (0):", int((df["is_minority"]==0).sum()),
      "| minority (1):", int((df["is_minority"]==1).sum()))
print("Unique RaceEth codes in use:", sorted(df["RaceEth"].unique().tolist()))
df[["who","RaceEth","is_minority","is_female","age"]].head()

In [ ]:
df

In [ ]:
# Move is_minority right after RaceEth
cols = list(df.columns)

# Remove is_minority and RaceEth from the list to rebuild cleanly
cols.remove("is_minority")
cols.remove("RaceEth")

# Reorder with RaceEth, is_minority first, then everything else
df = df[["RaceEth", "is_minority"] + cols]

df


In [ ]:
# Move 'who' to the front
cols = ["who"] + [c for c in df.columns if c != "who"]
df = df[cols]

df


In [ ]:
# stratified test sample: 100 samples, 58% majority / 42% minority
TEST_N = 100
MAJORITY_PCT = 0.58
MINORITY_PCT = 0.42

n_majority = int(TEST_N * MAJORITY_PCT)
n_minority = TEST_N - n_majority

test_majority = df[df["is_minority"] == 0].sample(n_majority, random_state=999)
test_minority = df[df["is_minority"] == 1].sample(n_minority, random_state=999)

test_df = pd.concat([test_majority, test_minority]).reset_index(drop=True)
train_pool = df[~df["who"].isin(test_df["who"])].reset_index(drop=True)

print("Test set shape:", test_df.shape)
print(test_df["is_minority"].value_counts())

In [ ]:
# assume:
# df = full dataset
# test_df = stratified test dataset
# both have column "who"

print("Original full dataset shape:", df.shape)
print("Test set shape:", test_df.shape)

# remove matching who IDs
train_pool = df[~df["who"].isin(test_df["who"])].reset_index(drop=True)

print("New train pool shape after removal:", train_pool.shape)
print("Rows removed:", df.shape[0] - train_pool.shape[0])


In [ ]:
train_pool.to_csv("/Users/ganeshjai/Documents/GitHub/Pipeline/data/train_pool_after_stratified_test_removal.csv", index=False)


In [ ]:
# df = original full dataset (or your updated dataset)
# test_df = stratified test dataset
# train_pool = df after removing those test IDs
id_col = "who"
overlap = set(test_df[id_col]) & set(train_pool[id_col])
print("Overlap count:", len(overlap))



In [ ]:
is_clean = train_pool[id_col].isin(test_df[id_col]).sum() == 0
print("No test IDs remain in train_pool:", is_clean)


In [ ]:
remaining = train_pool[train_pool[id_col].isin(test_df[id_col])]
print("Rows still in train_pool with test IDs:", remaining.shape)


In [ ]:
train_pool

In [ ]:
df

In [ ]:
pd.read_csv("")

In [ ]:
# build_from_training_pool.py
# Requires: rpy2, R, and the R package MatchIt
import os, json, zipfile, argparse
import numpy as np
import pandas as pd
import rpy2.robjects as robjects
from rpy2.robjects.packages import importr, PackageNotInstalledError
from rpy2.robjects import r, pandas2ri
from rpy2.robjects.conversion import localconverter

PRINT_SUMMARY = False

def _ensure_matchit_installed():
    try:
        importr("MatchIt")
    except PackageNotInstalledError:
        print("MatchIt not found. Installing...")
        r('install.packages("MatchIt", repos="https://cran.r-project.org")')
        importr("MatchIt")
        print("MatchIt installed.")

In [ ]:
# Cell 2 — Load data

train_pool = pd.read_csv("/Users/ganeshjai/Documents/GitHub/Pipeline/data/train_pool_after_stratified_test_removal.csv")
test_df    = pd.read_csv("/Users/ganeshjai/Documents/GitHub/Pipeline/data/test_holdout.csv")

id_col      = "who"
group_col   = "RaceEth"
majority_val = 1          # If 1 = majority in your data, keep this
match_cols  = ["age", "is_female"]  # later you will add psych_count here
dataset_size = 1000
seed = 42

train_pool



In [ ]:
import os
import zipfile
import numpy as np
import pandas as pd

# --- Optional deps for matching ---
from psmpy import PsmPy
import rpy2.robjects as robjects
from rpy2.robjects.packages import importr, PackageNotInstalledError
from rpy2.robjects.conversion import localconverter
from rpy2.robjects import r, pandas2ri

# ---------------------------
# Config — EDIT THESE
# ---------------------------
ID_COL = "who"                  # <-- your ID column
SPLIT_COL = "RaceEth"           # <-- or 'is_inpatient', etc.
MAJORITY_VALUE = 1              # <-- value representing majority in SPLIT_COL
MATCH_COLS = ("age", "is_female")
TOTAL_SIZE_PER_SET = 1000       # size of each output dataset
RATIOS = [i/10 for i in range(1, 11)]  # 0.1..1.0 (10%..100% minority)
PSM_TREATED_CAP = 100           # start small, scale later (250/500)
RANDOM_SEED = 42

# Where to save outputs
OUT_DIR = "/Users/ganeshjai/Documents/GitHub/Pipeline/data/ratio_sets"
ZIP_PATH = "/Users/ganeshjai/Documents/GitHub/Pipeline/data/ratio_sets_bundle.zip"  # optional zip

# ---------------------------
# Load your existing CSVs
# ---------------------------
train_pool = pd.read_csv("/Users/ganeshjai/Documents/GitHub/Pipeline/data/train_pool_after_stratified_test_removal.csv")
test_df    = pd.read_csv("/Users/ganeshjai/Documents/GitHub/Pipeline/data/test_holdout.csv")

# Safety: re-drop any leakage just in case
train_pool = train_pool[~train_pool[ID_COL].isin(test_df[ID_COL])].copy()

# ---------------------------
# Matching helpers
# ---------------------------
def _match_with_matchit(
    df,
    id_col,
    match_cols,
    treated_cap=100,
    treat_col="is_minority",
    print_summary=False
):
    import rpy2.robjects as robjects
    from rpy2.robjects import default_converter, pandas2ri
    from rpy2.robjects.conversion import localconverter
    from rpy2.robjects.packages import importr, PackageNotInstalledError

    # ---- hygiene & types ----
    safe_df = df.copy()
    for c in list(match_cols) + [id_col, treat_col]:
        if c not in safe_df.columns:
            raise ValueError(f"Column '{c}' not found in DataFrame.")
    # Make IDs strings (R handles them cleanly)
    safe_df[id_col] = safe_df[id_col].astype(str)

    # Optional: coerce numerics so R’s glm/probit behaves
    for c in match_cols:
        if safe_df[c].dtype == "O":
            safe_df[c] = pd.to_numeric(safe_df[c], errors="coerce")

    # ---- ensure MatchIt ----
    try:
        importr("MatchIt")
    except PackageNotInstalledError:
        print("MatchIt not installed. Installing from CRAN ...")
        robjects.r('install.packages("MatchIt", repos="https://cloud.r-project.org")')
        importr("MatchIt")

    # ---- pandas -> R inside localconverter ----
    with localconverter(default_converter + pandas2ri.converter):
        r_data = pandas2ri.py2rpy(safe_df)
    robjects.globalenv["inData"] = r_data

    formula_rhs = " + ".join(match_cols)

    # ---- R code (optimal matching, probit link, 1:2 ratio) ----
    robjects.r(f"""
        library(MatchIt)

        treated_subset <- inData[inData${treat_col} == 1, ]
        control_subset <- inData[inData${treat_col} == 0, ]

        if (nrow(treated_subset) > {treated_cap}) {{
          treated_subset <- treated_subset[1:{treated_cap}, ]
        }}
        subset_data <- rbind(treated_subset, control_subset)

        m.out <- matchit(
          {treat_col} ~ {formula_rhs},
          data   = subset_data,
          method = "optimal",
          distance = "glm",
          link   = "probit",
          ratio  = 2
        )

        matched <- match.data(m.out)
        matched <- matched[!is.na(matched$subclass), ]

        treated_ids <- matched[matched${treat_col} == 1, "{id_col}"]
        control_ids <- matched[matched${treat_col} == 0, "{id_col}"]
        summ_txt <- capture.output(summary(m.out, un = FALSE))
    """)

    if print_summary:
        summ = robjects.r("summ_txt")
        print("\n".join(list(summ)))

    # ---- R -> pandas inside localconverter ----
    with localconverter(default_converter + pandas2ri.converter):
        treated_ids = pandas2ri.rpy2py(robjects.r("treated_ids"))
        control_ids = pandas2ri.rpy2py(robjects.r("control_ids"))

    min_ids = pd.Series(pd.unique(pd.Series(treated_ids)), name=id_col)
    maj_ids = pd.Series(pd.unique(pd.Series(control_ids)), name=id_col)
    return min_ids, maj_ids


def get_psm_pools(
    train_df,
    id_col,
    split_col,
    majority_value,
    match_cols,
    treated_cap=100,
    seed=RANDOM_SEED
):
    # Work on a copy; add treatment flag
    df = train_df.copy()

    # Ensure ID dtype matches test_df’s ID dtype for the leakage check later
    if id_col in test_df.columns:
        df[id_col] = df[id_col].astype(test_df[id_col].dtype)

    df["is_minority"] = (df[split_col] != majority_value).astype(int)

    # --- Run MatchIt (optimal, probit, 1:2) ---
    min_ids, maj_ids = _match_with_matchit(
        df=df,
        id_col=id_col,
        match_cols=match_cols,
        treated_cap=treated_cap,
        treat_col="is_minority",
        print_summary=False
    )

    # Map back to full rows
    min_pool = df[df[id_col].isin(min_ids)].drop(columns=["is_minority"])
    maj_pool = df[df[id_col].isin(maj_ids)].drop(columns=["is_minority"])

    # Extra guard: remove any possible test leakage
    if id_col in test_df.columns:
        min_pool = min_pool[~min_pool[id_col].isin(test_df[id_col])]
        maj_pool = maj_pool[~maj_pool[id_col].isin(test_df[id_col])]

    # Deduplicate by ID
    min_pool = min_pool.drop_duplicates(subset=[id_col])
    maj_pool = maj_pool.drop_duplicates(subset=[id_col])

    return min_pool, maj_pool



'''
def _match_with_psmpy(df, id_col, match_cols, treat_col="is_minority", treated_cap=100):
    # Approximate 1:2 using PsmPy's 12n matcher
    cols_exclude = list(df.columns.difference(list(match_cols) + [treat_col]).drop(id_col))
    psm = PsmPy(df, treatment=treat_col, indx=id_col, exclude=cols_exclude)
    psm.logistic_ps()
    psm.knn_matched_12n(matcher='propensity_logit', how_many=2)

    matched = psm.matched_ids.copy()
    treated_col = matched.columns[0]
    control_cols = matched.columns[1:]

    min_ids = pd.Series(pd.unique(matched[treated_col].dropna()), name=id_col)
    maj_ids = pd.Series(pd.unique(pd.concat([matched[c] for c in control_cols], ignore_index=True).dropna()), name=id_col)

    if len(min_ids) > treated_cap:
        min_ids = min_ids.iloc[:treated_cap].copy()
    return min_ids, maj_ids

'''


# ---------------------------
# Build ratioed datasets
# ---------------------------
def build_ratio_sets(min_pool, maj_pool, id_col, total_size=1000, ratios=None, seed=RANDOM_SEED):
    rng = np.random.RandomState(seed)
    ratios = ratios or [i/10 for i in range(1, 11)]
    out = {}

    for r in ratios:
        n_min = int(round(r * total_size))
        n_maj = total_size - n_min

        # sample with or without replacement depending on pool sizes
        min_replace = len(min_pool) < n_min
        maj_replace = len(maj_pool) < n_maj

        min_sample = min_pool.sample(n=n_min, replace=min_replace, random_state=rng)
        maj_sample = maj_pool.sample(n=n_maj, replace=maj_replace, random_state=rng)

        df_r = pd.concat([min_sample, maj_sample], ignore_index=True)
        df_r = df_r.sample(frac=1.0, random_state=rng).reset_index(drop=True)
        out[f"{int(r*100)}%"] = df_r

    return out


# ---------------------------
# Run
# ---------------------------
os.makedirs(OUT_DIR, exist_ok=True)

minority_pool, majority_pool = get_psm_pools(
    train_df=train_pool,
    id_col=ID_COL,
    split_col=SPLIT_COL,
    majority_value=MAJORITY_VALUE,
    match_cols=MATCH_COLS,
    treated_cap=PSM_TREATED_CAP,
    seed=RANDOM_SEED
)

ratio_sets = build_ratio_sets(
    min_pool=minority_pool,
    maj_pool=majority_pool,
    id_col=ID_COL,
    total_size=TOTAL_SIZE_PER_SET,
    ratios=RATIOS,
    seed=RANDOM_SEED
)

# Save each dataset
saved_paths = []
for label, df_out in ratio_sets.items():
    fname = f"train_{label}_minority_size{TOTAL_SIZE_PER_SET}.csv"
    fpath = os.path.join(OUT_DIR, fname)
    df_out.to_csv(fpath, index=False)
    saved_paths.append(fpath)

# (Optional) save a single zip bundle of the 10 datasets
with zipfile.ZipFile(ZIP_PATH, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for p in saved_paths:
        zf.write(p, arcname=os.path.basename(p))

print("Done.")
print("Minority pool size:", len(minority_pool), "| Majority pool size:", len(majority_pool))
print("Saved CSVs in:", OUT_DIR)
print("Zip bundle at:", ZIP_PATH)


In [ ]:
print(train_pool[SPLIT_COL].value_counts(dropna=False))
print("Minority in train_pool:", (train_pool[SPLIT_COL] != MAJORITY_VALUE).sum())
print("Majority in train_pool:", (train_pool[SPLIT_COL] == MAJORITY_VALUE).sum())


In [ ]:
print(len(minority_pool), len(majority_pool))
